# Eval checkpoint trên bộ đã đóng băng
Sinh output greedy trên dev (hoặc test, chỉ sau khi khóa giao thức) và bộ P, chấm bằng luật, ghi `results/<run>/<checkpoint>/` (PLAN 3.6); thêm perplexity và KL so với base trên wikitext-2 (PLAN 3.3). Mặc định chạy E0: base zero-shot, base few-shot và 360M-Instruct bằng template riêng của nó. Nên dùng runtime GPU.


## 1. Cài thư viện

In [ ]:
%pip install -q transformers==5.20.2 huggingface-hub==1.32.0 psutil==7.2.2 safetensors==0.8.0 pyyaml==6.0.3
from importlib.metadata import version
wanted = {"transformers": "5.20.2", "huggingface-hub": "1.32.0", "psutil": "7.2.2", "safetensors": "0.8.0", "pyyaml": "6.0.3"}
installed = {name: version(name) for name in wanted}
print(installed)
assert installed == wanted, "Phiên bản chưa khớp requirements.txt; chạy lại cell này."


## 2. Nạp mã nguồn của dự án

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "bfa69889986f7080847997e244c6d605b59abfad2b375ff9429f490a553b260f"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "bfa69889986f7080847997e244c6d605b59abfad2b375ff9429f490a553b260f")
print("Working directory:", root)


## 3. Kiểm tra mã

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr)
checks.check_returncode()


## 4. Đăng nhập HF Hub

Cần secret `HF_TOKEN` trong 🔑 **Secrets** của Colab, đã bật **Notebook access**. Chạy cell này **bằng nút ▶ trên giao diện Colab**: secret không đọc được khi cell được chạy qua MCP.

Điền `DATASET_REPO` dạng `<username>/<tên repo>`: dataset private chứa `eval_sets/`.

In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
DATASET_REPO = ""  # ví dụ "your-username/sft-capability-data"
print("Logged in as:", whoami()["name"])
assert DATASET_REPO, "Điền DATASET_REPO trước khi chạy tiếp."


## 5. Tải dev/test và kiểm hash
Dừng ở đây nếu `src.eval.manifest` báo file thiếu hoặc khác.

In [ ]:
from src.eval.run_eval import fetch
fetch(DATASET_REPO)
!python -m src.eval.manifest


## 6. Sinh output và chấm
`MODELS` gồm (tên checkpoint, model, tùy chọn). Model là khóa trong `configs/models.yaml`, id trên Hub hoặc thư mục checkpoint. Mỗi dòng in thời gian sinh để đo ngân sách eval (PLAN 3.6).

In [ ]:
import torch
RUN, SPLIT = "e0", "dev"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = [("base-zero-shot", "base", ""), ("base-few-shot", "base", "--few-shot"),
          ("instruct-360m", "chat_reference", "--template native")]
for name, model, options in MODELS:
    !python -m src.eval.run_eval --model {model} --run {RUN} --checkpoint {name} --split {SPLIT} --device {DEVICE} {options}


## 7. Perplexity và KL trên wikitext-2
KL tính so với base; với chính base thì bỏ qua KL.

In [ ]:
for model in ("base", "chat_reference"):
    !python -m src.eval.knowledge --model {model} --device {DEVICE} --output results/{RUN}/knowledge-{model}.json
